Advanced RAG Techniques!

Let's start by digging into ingest:

1. No LangChain! Just native for maximum flexibility
2. Let's use an LLM to divide up chunks in a sensible way
3. Let's use the best chunk size and encoder from yesterday
4. Let's also have the LLM rewrite chunks in a way that's most useful ("document pre-processing")

In [ ]:
from pathlib import Path
from openai import OpenAI
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from chromadb import PersistentClient
from tqdm import tqdm
from litellm import completion
import numpy as np
from sklearn.manifold import TSNE
import plotly.graph_objects as go
import os

load_dotenv(override=True)
openai = OpenAI()

google_api_key = os.getenv('GOOGLE_API_KEY')
gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"
gemini = OpenAI(api_key=google_api_key,base_url=gemini_url)

ollama_url = "http://localhost:11434/v1"
ollama = OpenAI(api_key="ollama",base_url=ollama_url)

groq_api_key = os.getenv('GROQ_API_KEY')
groq_url = "https://api.groq.com/openai/v1"
groq = OpenAI(api_key=groq_api_key,base_url=groq_url)

# MODEL = "groq/llama-3.1-8b-instant"
MODEL = "groq/openai/gpt-oss-120b"
# MODEL = "llama-3.1-8b-instant"
# MODEL = "gemma2:2b"
# MODEL = "gemini-2.5-flash-lite"

DB_NAME = "preprocessed_db"
collection_name = "docs"
embedding_model = "all-MiniLM-L6-v2"
KNOWLEDGE_BASE_PATH = Path("knowledge-base")
AVERAGE_CHUNK_SIZE = 500

In [2]:
# Inspired by LangChain's Document - let's have something similar

class Result(BaseModel):
    page_content: str
    metadata: dict

In [3]:
# A class to perfectly represent a chunk

class Chunk(BaseModel):
    headline: str = Field(description="A brief heading for this chunk, typically a few words, that is most likely to be surfaced in a query")
    summary: str = Field(description="A few sentences summarizing the content of this chunk to answer common questions")
    original_text: str = Field(description="The original text of this chunk from the provided document, exactly as is, not changed in any way")

    def as_result(self, document):
        metadata = {"source": document["source"], "type": document["type"]}
        return Result(page_content=self.headline + "\n\n" + self.summary + "\n\n" + self.original_text,metadata=metadata)


class Chunks(BaseModel):
    chunks: list[Chunk]

## Three steps:

1. Fetch documents from the knowledge base, like LangChain did
2. Call an LLM to turn documents into Chunks
3. Store the Chunks in Chroma

That's it!

### Let's start with Step 1

In [4]:
def fetch_documents():
    """A homemade version of the LangChain DirectoryLoader"""

    documents = []

    for folder in KNOWLEDGE_BASE_PATH.iterdir():
        doc_type = folder.name
        for file in folder.rglob("*.md"):
            with open(file, "r", encoding="utf-8") as f:
                documents.append({"type": doc_type, "source": file.as_posix(), "text": f.read()})

    print(f"Loaded {len(documents)} documents")
    return documents

In [5]:
documents = fetch_documents()
documents[0]

Loaded 76 documents


{'type': 'company',
 'source': 'knowledge-base/company/about.md',
 'text': "# About Insurellm\n\nInsurellm was founded by Avery Lancaster in 2015 as an insurance tech startup designed to disrupt an industry in need of innovative products. Its first product was Markellm, the marketplace connecting consumers with insurance providers.\n\nThe company experienced rapid growth in its first five years, expanding its product portfolio to include Carllm (auto insurance portal), Homellm (home insurance portal), and Rellm (enterprise reinsurance platform). By 2020, Insurellm had reached a peak of 200 employees with 12 offices across the US.\n\nHowever, the company underwent a strategic restructuring in 2022-2023 to focus on profitability and sustainable growth. This included consolidating office locations, implementing a remote-first strategy, and streamlining operations. As of 2025, Insurellm operates with a lean, highly efficient team of 32 employees who have built a portfolio of 32 active cont

### Step 2 - make the chunks

In [6]:
# def make_prompt(document):
#     how_many = (len(document["text"]) // AVERAGE_CHUNK_SIZE) + 1
#     return f"""
# You take a document and you split the document into overlapping chunks for a KnowledgeBase.

# The document is from the shared drive of a company called Insurellm.
# The document is of type: {document["type"]}
# The document has been retrieved from: {document["source"]}

# A chatbot will use these chunks to answer questions about the company.
# You should divide up the document as you see fit, being sure that the entire document is returned in the chunks - don't leave anything out.
# This document should probably be split into {how_many} chunks, but you can have more or less as appropriate.
# There should be overlap between the chunks as appropriate; typically about 25% overlap or about 50 words, so you have the same text in multiple chunks for best retrieval results.

# For each chunk, you should provide a headline, a summary, and the original text of the chunk.
# Together your chunks should represent the entire document with overlap.

# Here is the document:

# {document["text"]}

# Respond with the chunks.
# """

def make_prompt(document):
    how_many = (len(document["text"]) // AVERAGE_CHUNK_SIZE) + 1
    return f"""
You take a document and you split the document into overlapping chunks for a KnowledgeBase.

... (keep everything else the same) ...

CRITICAL STRUCTURAL VALIDATION RULE:
Your response must strictly follow this exact JSON structure layout with a single outer dictionary key:
{{
  "chunks": [
    {{
      "headline": "Example Headline 1",
      "summary": "Example summary text...",
      "original_text": "Example clean original text..."
    }}
  ]
}}

STRICT STRING CHARACTER RULES:
1. DO NOT use curly smart quotes or curly apostrophes (like ’ or ‘). Use ONLY standard plain-text straight apostrophes (') if necessary, or omit them entirely (e.g., write "Insurellms" instead of "Insurellm's").
2. Clean the text completely: remove all markdown symbols like '**', '#', and '-'.
3. DO NOT duplicate the "chunks" key at the root.

Here is the document:
{document["text"]}

Respond with the chunks object.
"""

In [7]:
print(make_prompt(documents[0]))


You take a document and you split the document into overlapping chunks for a KnowledgeBase.

... (keep everything else the same) ...

CRITICAL STRUCTURAL VALIDATION RULE:
Your response must strictly follow this exact JSON structure layout with a single outer dictionary key:
{
  "chunks": [
    {
      "headline": "Example Headline 1",
      "summary": "Example summary text...",
      "original_text": "Example clean original text..."
    }
  ]
}

STRICT STRING CHARACTER RULES:
1. DO NOT use curly smart quotes or curly apostrophes (like ’ or ‘). Use ONLY standard plain-text straight apostrophes (') if necessary, or omit them entirely (e.g., write "Insurellms" instead of "Insurellm's").
2. Clean the text completely: remove all markdown symbols like '**', '#', and '-'.
3. DO NOT duplicate the "chunks" key at the root.

Here is the document:
# About Insurellm

Insurellm was founded by Avery Lancaster in 2015 as an insurance tech startup designed to disrupt an industry in need of innovative

In [8]:
def make_messages(document):
    return [
        {"role": "user", "content": make_prompt(document)},
    ]

In [9]:
make_messages(documents[0])

[{'role': 'user',
  'content': '\nYou take a document and you split the document into overlapping chunks for a KnowledgeBase.\n\n... (keep everything else the same) ...\n\nCRITICAL STRUCTURAL VALIDATION RULE:\nYour response must strictly follow this exact JSON structure layout with a single outer dictionary key:\n{\n  "chunks": [\n    {\n      "headline": "Example Headline 1",\n      "summary": "Example summary text...",\n      "original_text": "Example clean original text..."\n    }\n  ]\n}\n\nSTRICT STRING CHARACTER RULES:\n1. DO NOT use curly smart quotes or curly apostrophes (like ’ or ‘). Use ONLY standard plain-text straight apostrophes (\') if necessary, or omit them entirely (e.g., write "Insurellms" instead of "Insurellm\'s").\n2. Clean the text completely: remove all markdown symbols like \'**\', \'#\', and \'-\'.\n3. DO NOT duplicate the "chunks" key at the root.\n\nHere is the document:\n# About Insurellm\n\nInsurellm was founded by Avery Lancaster in 2015 as an insurance t

In [ ]:
# def process_document(document):
#     messages = make_messages(document)
#     response = groq.chat.completions.create(
#         model=MODEL, 
#         messages=messages, 
#         response_format={"type": "json_object"},
#         max_tokens=4000  # 🌟 Unlocks a massive text generation ceiling
#     )
#     reply = response.choices[0].message.content
#     doc_as_chunks = Chunks.model_validate_json(reply).chunks
#     return [chunk.as_result(document) for chunk in doc_as_chunks]

def process_document(document):
    safe_doc = document.copy()
    # Keep document sizes safe for the free tier token bucket limits
    if len(safe_doc['text']) > 18000:
        safe_doc['text'] = safe_doc['text'][:18000]

    messages = make_messages(safe_doc)
    
    # 🌟 USE LITELLM NATIVE STRUCTURED OUTPUTS
    # Passing your Pydantic class directly here forces clean JSON validation 
    # on the server without background retry-loops breaking your rate limits.
    response = completion(
        model=MODEL, 
        messages=messages, 
        response_format=Chunks,  # Pass the Pydantic class object directly here!
        max_tokens=3000
    )
    
    # LiteLLM automatically parses the JSON into a native Pydantic object
    # inside response._hidden_params["predicted_to_pydantic"] or parses the content string safely
    reply = response.choices[0].message.content
    
    doc_as_chunks = Chunks.model_validate_json(reply).chunks
    return [chunk.as_result(document) for chunk in doc_as_chunks]

In [ ]:
process_document(documents[0])

In [ ]:
# def create_chunks(documents):
#     chunks = []
#     for doc in tqdm(documents):
#         chunks.extend(process_document(doc))
#     return chunks

from litellm.exceptions import RateLimitError, BadRequestError
import time
from litellm.exceptions import RateLimitError
from tqdm import tqdm
import instructor

def create_chunks_resilient(documents, start_index=16):
    """
    start_index=16 means we are skipping the first 16 documents (0 to 15) 
    that you already successfully processed before the freeze.
    """
    processed_chunks = []
    print(f"🚀 Resuming pipeline safely. Processing from document {start_index + 1}/76...")
    
    # Slice the documents list to start exactly where you left off
    for i, doc in enumerate(documents[start_index:], start_index + 1):
        rate_limit_retries = 0  # Track retries for THIS specific document
        
        while True:
            try:
                processed_docs = process_document(doc)
                processed_chunks.extend(processed_docs)
                time.sleep(3.0)  # Safe pacing buffer
                break            # Success! Move to next document
                
            except RateLimitError:
                rate_limit_retries += 1
                
                # TRAP BREAKER: If it hits a rate limit twice on the SAME file, skip it!
                if rate_limit_retries >= 2:
                    print(f"\n🚨 [Doc {i}/76] is too large for the Groq Free Tier token bucket. Skipping to break the trap.")
                    break
                    
                print(f"\n⚠️ Groq token ceiling reached at doc [{i}/76]. Waiting 65s for full bucket reset...")
                time.sleep(65)
                continue 
                
            except BadRequestError as e:
                print(f"\n❌ JSON compilation failed for: {doc['source']}. Skipping file.")
                break 
                
            except Exception as e:
                print(f"\n❌ Unexpected processing error on {doc['source']}: {e}")
                break
                
    return processed_chunks

# Run this to pick up right at document 17!
chunks = create_chunks_resilient(documents, start_index=16)
# Execute this block to process your data pipeline cleanly!
# chunks = create_chunks(documents)

c:\Users\HP\Udemy\Ai\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\HP\Udemy\Ai\venv\Lib\site-packages\instructor\providers\gemini\client.py:5: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai  # type: ignore[import-not-found]


🚀 Resuming pipeline safely. Processing from document 17/76...

Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.


Provider List: https://docs.litellm.ai/docs/providers


⚠️ Groq token ceiling reached at doc [17/76]. Waiting 65s for full bucket reset...

Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.


Provider List: https://docs.litellm.ai/docs/providers


🚨 [Doc 17/76] is too large for the Groq Free Tier token bucket. Skipping to break the trap.


c:\Users\HP\Udemy\Ai\venv\Lib\site-packages\pydantic\main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected 9 fields but got 5: Expected `Message` - serialized value may not be as expected [field_name='choices', input_value=Message(content='{"chunks...er_specific_fields=None), input_type=Message])
  PydanticSerializationUnexpectedValue(Expected `StreamingChoices` - serialized value may not be as expected [field_name='choices', input_value=Choices(finish_reason='to...r_specific_fields=None)), input_type=Choices])
  return self.__pydantic_serializer__.to_python(



Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.


Provider List: https://docs.litellm.ai/docs/providers


⚠️ Groq token ceiling reached at doc [19/76]. Waiting 65s for full bucket reset...

Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.


Provider List: https://docs.litellm.ai/docs/providers


❌ JSON compilation failed for: knowledge-base/contracts/Contract with Greenstone Insurance for Homellm.md. Skipping file.

Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.


Provider List: https://docs.litellm.ai/docs/providers


⚠️ Groq token ceiling reached at doc [20/76]. Waiting 65s for full bucket reset...

Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.I

c:\Users\HP\Udemy\Ai\venv\Lib\site-packages\pydantic\main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected 9 fields but got 5: Expected `Message` - serialized value may not be as expected [field_name='choices', input_value=Message(content=None, rol...er_specific_fields=None), input_type=Message])
  PydanticSerializationUnexpectedValue(Expected `StreamingChoices` - serialized value may not be as expected [field_name='choices', input_value=Choices(finish_reason='to...r_specific_fields=None)), input_type=Choices])
  return self.__pydantic_serializer__.to_python(



Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.


Provider List: https://docs.litellm.ai/docs/providers


❌ JSON compilation failed for: knowledge-base/employees/Robert Chen.md. Skipping file.

Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.


Provider List: https://docs.litellm.ai/docs/providers


⚠️ Groq token ceiling reached at doc [65/76]. Waiting 65s for full bucket reset...

Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.


Provider List: https://docs.litellm.ai/docs/providers


❌ JSON compilation failed for: knowledge-base/employees/Samantha Greene.md. Skipping file.

Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug t

In [13]:
print(f"📦 Total semantic chunks in memory: {len(chunks)}")

📦 Total semantic chunks in memory: 80


In [14]:
if len(chunks) > 0:
    print(f"🏁 Last successfully processed file: {chunks[-1].metadata.get('source')}")
else:
    print("The chunks list is empty. Try running the cell again.")

🏁 Last successfully processed file: knowledge-base/products/Markellm.md


### Finally, Step 3 - save the embeddings

In [15]:
# def create_embeddings(chunks):
#     chroma = PersistentClient(path=DB_NAME)
#     if collection_name in [c.name for c in chroma.list_collections()]:
#         chroma.delete_collection(collection_name)

#     texts = [chunk.page_content for chunk in chunks]
#     emb = openai.embeddings.create(model=embedding_model, input=texts).data
#     vectors = [e.embedding for e in emb]

#     collection = chroma.get_or_create_collection(collection_name)

#     ids = [str(i) for i in range(len(chunks))]
#     metas = [chunk.metadata for chunk in chunks]

#     collection.add(ids=ids, embeddings=vectors, documents=texts, metadatas=metas)
#     print(f"Vectorstore created with {collection.count()} documents")

from chromadb import PersistentClient
from chromadb.utils import embedding_functions

def create_embeddings(chunks):
    chroma = PersistentClient(path=DB_NAME)
    
    # Reset collection if it already exists
    if collection_name in [c.name for c in chroma.list_collections()]:
        chroma.delete_collection(collection_name)

    # 🌟 Load the local all-MiniLM-L6-v2 model wrapper inside Chroma
    sentence_transformer_ef = embedding_functions.SentenceTransformerEmbeddingFunction(
        model_name="all-MiniLM-L6-v2"
    )

    # Pass the embedding function directly when creating the collection
    collection = chroma.get_or_create_collection(
        name=collection_name, 
        embedding_function=sentence_transformer_ef
    )

    # Extract texts and metadata matching your custom chunk output structure
    # NOTE: Your chunks are custom Result objects, so access attributes directly
    texts = [chunk.page_content for chunk in chunks]
    metas = [chunk.metadata for chunk in chunks]
    ids = [str(i) for i in range(len(chunks))]

    # 🌟 Chroma automatically computes the embeddings using the local model here!
    collection.add(
        ids=ids, 
        documents=texts, 
        metadatas=metas
    )
    
    print(f"✅ Vectorstore created with {collection.count()} documents using local all-MiniLM-L6-v2!")

In [16]:
create_embeddings(chunks)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1256.81it/s]


✅ Vectorstore created with 80 documents using local all-MiniLM-L6-v2!


## VISUALIZE

In [17]:
chroma = PersistentClient(path=DB_NAME)
collection = chroma.get_or_create_collection(collection_name)
result = collection.get(include=['embeddings', 'documents', 'metadatas'])
vectors = np.array(result['embeddings'])
documents = result['documents']
metadatas = result['metadatas']
doc_types = [metadata['type'] for metadata in metadatas]
colors = [['blue', 'green', 'red', 'orange'][['products', 'employees', 'contracts', 'company'].index(t)] for t in doc_types]

In [18]:
tsne = TSNE(n_components=2, random_state=42)
reduced_vectors = tsne.fit_transform(vectors)

# Create the 2D scatter plot
fig = go.Figure(data=[go.Scatter(
    x=reduced_vectors[:, 0],
    y=reduced_vectors[:, 1],
    mode='markers',
    marker=dict(size=5, color=colors, opacity=0.8),
    text=[f"Type: {t}<br>Text: {d[:100]}..." for t, d in zip(doc_types, documents)],
    hoverinfo='text'
)])

fig.update_layout(title='2D Chroma Vector Store Visualization',
    scene=dict(xaxis_title='x',yaxis_title='y'),
    width=800,
    height=600,
    margin=dict(r=20, b=10, l=10, t=40)
)

fig.show()

In [19]:
tsne = TSNE(n_components=3, random_state=42)
reduced_vectors = tsne.fit_transform(vectors)

# Create the 3D scatter plot
fig = go.Figure(data=[go.Scatter3d(
    x=reduced_vectors[:, 0],
    y=reduced_vectors[:, 1],
    z=reduced_vectors[:, 2],
    mode='markers',
    marker=dict(size=5, color=colors, opacity=0.8),
    text=[f"Type: {t}<br>Text: {d[:100]}..." for t, d in zip(doc_types, documents)],
    hoverinfo='text'
)])

fig.update_layout(
    title='3D Chroma Vector Store Visualization',
    scene=dict(xaxis_title='x', yaxis_title='y', zaxis_title='z'),
    width=900,
    height=700,
    margin=dict(r=10, b=10, l=10, t=40)
)

fig.show()

## let's build an Advanced RAG!

We will use these techniques:

1. Reranking - reorder the rank results
2. Query re-writing

In [20]:
class RankOrder(BaseModel):
    order: list[int] = Field(
        description="The order of relevance of chunks, from most relevant to least relevant, by chunk id number"
    )

In [ ]:
# def rerank(question, chunks):
#     system_prompt = """
# You are a document re-ranker.
# You are provided with a question and a list of relevant chunks of text from a query of a knowledge base.
# The chunks are provided in the order they were retrieved; this should be approximately ordered by relevance, but you may be able to improve on that.
# You must rank order the provided chunks by relevance to the question, with the most relevant chunk first.
# Reply only with the list of ranked chunk ids, nothing else. Include all the chunk ids you are provided with, reranked.
# """
#     user_prompt = f"The user has asked the following question:\n\n{question}\n\nOrder all the chunks of text by relevance to the question, from most relevant to least relevant. Include all the chunk ids you are provided with, reranked.\n\n"
#     user_prompt += "Here are the chunks:\n\n"
#     for index, chunk in enumerate(chunks):
#         user_prompt += f"# CHUNK ID: {index + 1}:\n\n{chunk.page_content}\n\n"
#     user_prompt += "Reply only with the list of ranked chunk ids, nothing else."
#     messages = [
#         {"role": "system", "content": system_prompt},
#         {"role": "user", "content": user_prompt},
#     ]
#     response = completion(model=MODEL, messages=messages, response_format=RankOrder)
#     reply = response.choices[0].message.content
#     order = RankOrder.model_validate_json(reply).order
#     print(order)
#     return [chunks[i - 1] for i in order]

In [75]:
def rerank(question, chunks):
    if len(chunks) <= 10:
        return _execute_llm_rerank(question, chunks)
        
    print(f"🧩 Splitting {len(chunks)} chunks into safe structural batches...")
    
    # 1. Batch into pools of 10 to protect token ceilings
    batch_size = 10
    batches = [chunks[i:i + batch_size] for i in range(0, len(chunks), batch_size)]
    
    ranked_pools = []
    for batch in batches:
        ranked_pools.append(_execute_llm_rerank(question, batch))
        
    # 2. Extract the top 4 structural winners from each batch 
    final_contenders = []
    for pool in ranked_pools:
        final_contenders.extend(pool[:4])
        
    # 3. Final round uses the exact same isolated execution logic safely!
    print("🏆 Running final validation sort on top contenders...")
    final_ranked_leaders = _execute_llm_rerank(question, final_contenders)
    
    # 4. Clean up any omitted fragments so no data drops
    leader_ids = {id(c) for c in final_ranked_leaders}
    remaining_elements = [c for c in chunks if id(c) not in leader_ids]
    
    return final_ranked_leaders + remaining_elements


def _execute_llm_rerank(question, chunk_pool):
    """
    Reranks documents using static memory address lookup strings to eliminate structural index shifting.
    """
    # Create a completely unique string mapping dictionary using memory address keys
    lookup_map = {str(id(chunk)): chunk for chunk in chunk_pool}
    
    system_prompt = """
You are a strict data-processing utility designed to rank text fragments. 
You are provided with a question and a list of text chunks, each labeled with a Unique ID string. 
Your single job is to return a valid JSON object containing an array of those exact Unique ID strings sorted by relevance to the question.

You MUST follow this exact JSON output structure:
{
  "order": ["ID_STRING_1", "ID_STRING_2", "ID_STRING_3"]
}

CRITICAL RULES:
1. The only permitted root key in your JSON string is "order".
2. Copy the exact Unique ID strings provided to you. Do not shorten them or use integers.
3. Return ONLY the JSON object. No explanations.
"""
    
    user_prompt = f"Question: {question}\n\nChunks to process:\n"
    for chunk_id, chunk in lookup_map.items():
        user_prompt += f"Unique ID: {chunk_id}\nContent: {chunk.page_content}\n\n"
        
    user_prompt += "Output the final JSON string:"
    
    attempts = 0
    while True:
        try:
            attempts += 1
            response = completion(
                model=MODEL, 
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": user_prompt}
                ], 
                response_format={"type": "json_object"} 
            )
            reply = response.choices[0].message.content
            data = json.loads(reply)
            raw_order = data.get("order")
            
            if not raw_order or not isinstance(raw_order, list):
                raise ValueError("Missing 'order' key structure.")
            
            # Map back to the original objects safely using our lookup dictionary keys
            final_ranked_pool = []
            for item in raw_order:
                clean_id = str(item).strip()
                if clean_id in lookup_map and lookup_map[clean_id] not in final_ranked_pool:
                    final_ranked_pool.append(lookup_map[clean_id])
            
            # Append any fragments the model left out
            for chunk in chunk_pool:
                if chunk not in final_ranked_pool:
                    final_ranked_pool.append(chunk)
                    
            return final_ranked_pool
            
        except Exception as e:
            if attempts >= 3:
                print(f"🚨 [Fallback] Preserving default sequence on attempt {attempts}.")
                return chunk_pool 
            time.sleep(2.0)
            continue

In [26]:
# RETRIEVAL_K = 10

# def fetch_context_unranked(question):
#     query = openai.embeddings.create(model=embedding_model, input=[question]).data[0].embedding
#     results = collection.query(query_embeddings=[query], n_results=RETRIEVAL_K)
#     chunks = []
#     for result in zip(results["documents"][0], results["metadatas"][0]):
#         chunks.append(Result(page_content=result[0], metadata=result[1]))
#     return chunks

from sentence_transformers import SentenceTransformer

RETRIEVAL_K = 10

def fetch_context_unranked(question):
    chroma = PersistentClient(path=DB_NAME)
    collection = chroma.get_collection(name=collection_name)
    
    # 1. Load your local transformer model architecture
    model = SentenceTransformer("all-MiniLM-L6-v2")
    
    # 2. Vectorize the text query locally into a numerical list array
    query_vector = model.encode(question).tolist()
    
    # 3. Match against stored arrays inside Chroma DB
    results = collection.query(
        query_embeddings=[query_vector], 
        n_results=RETRIEVAL_K
    )
    
    # 4. Return formatted Result chunk structures
    chunks = []
    for text, metadata in zip(results["documents"][0], results["metadatas"][0]):
        chunks.append(Result(page_content=text, metadata=metadata))
        
    return chunks


In [27]:
question = "Who won the IIOTY award?"
chunks = fetch_context_unranked(question)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1944.50it/s]


In [28]:
for chunk in chunks:
    print(chunk.page_content[:15]+"...")

Annual Performa...
HR Record

Mich...
Other HR Notes
...
Michael O’Brien...
Summary

Date o...
Compensation Hi...
Emily Tran’s Pe...
Summary

Michae...
AI-Powered Matc...
HR Record

This...


In [29]:
reranked = rerank(question, chunks)

[5, 7, 10, 1, 6, 4, 2, 3, 8, 9, 3]


c:\Users\HP\Udemy\Ai\venv\Lib\site-packages\pydantic\main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected 9 fields but got 5: Expected `Message` - serialized value may not be as expected [field_name='choices', input_value=Message(content='{"order"...er_specific_fields=None), input_type=Message])
  PydanticSerializationUnexpectedValue(Expected `StreamingChoices` - serialized value may not be as expected [field_name='choices', input_value=Choices(finish_reason='to...r_specific_fields=None)), input_type=Choices])
  return self.__pydantic_serializer__.to_python(


In [30]:
for chunk in reranked:
    print(chunk.page_content[:15]+"...")

Summary

Date o...
Emily Tran’s Pe...
HR Record

This...
Annual Performa...
Compensation Hi...
Michael O’Brien...
HR Record

Mich...
Other HR Notes
...
Summary

Michae...
AI-Powered Matc...
Other HR Notes
...


In [31]:
question = "Who went to Manchester University?"
RETRIEVAL_K = 20
chunks = fetch_context_unranked(question)
for index, c in enumerate(chunks):
    if "manchester" in c.page_content.lower():
        print(index)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1374.13it/s]


6


In [40]:
reranked = rerank(question, chunks)

🧩 Splitting 20 chunks into smaller pools to protect Groq TPM ceilings...
   Sorting Batch 1/2...
   Sorting Batch 2/2...
🏆 Running final validation sort on top contenders...

Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.


Provider List: https://docs.litellm.ai/docs/providers

⚠️ Clean structural parse failed (Attempt 1/3). Resetting in 2s...

Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.


Provider List: https://docs.litellm.ai/docs/providers

⚠️ Clean structural parse failed (Attempt 2/3). Resetting in 2s...

Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.


Provider List: https://docs.litellm.ai/docs/providers

🚨 [Fallback Activated] Couldn't parse LLM ranking on attempt 3. P

In [42]:
for index, c in enumerate(reranked):
    if "manchester" in c.page_content.lower():
        print(index)

2


In [44]:
reranked[2].page_content

'HR Record\n\nSummary of Jessica Liu\\u2019s career and achievements\n\nHR Record\\n\\nJessica Liu\\n\\n# Summary\\n- Date of Birth: April 30, 1996\\n- Job Title: Frontend Developer\\n- Location: Remote (Based in Seattle, Washington)\\n- Current Salary: $92,000\\n\\n# Insurellm Career Progression\\n- July 2022 \\u2013 Present: Frontend Developer\\n  - Develops user interfaces for Rellm reinsurance platform using React\\n  - Implements responsive designs and ensures cross-browser compatibility\\n  - Collaborates with UX designers and backend engineers\\n- January 2020 \\u2013 June 2022: Junior Frontend Developer\\n  - Built UI components for internal tools and customer-facing applications\\n  - Fixed bugs and improved performance of existing web applications\\n  - Participated in code reviews and learned best practices\\n- June 2018 \\u2013 December 2019: Web Developer Intern at StartupLabs\\n  - Created landing pages and marketing websites\\n  - Learned HTML, CSS, JavaScript, and React

In [76]:
def fetch_context(question):
    chunks = fetch_context_unranked(question)
    return rerank(question, chunks)

In [77]:
SYSTEM_PROMPT = """
You are a knowledgeable, friendly assistant representing the company Insurellm.
You are chatting with a user about Insurellm.
Your answer will be evaluated for accuracy, relevance and completeness, so make sure it only answers the question and fully answers it.
If you don't know the answer, say so.
For context, here are specific extracts from the Knowledge Base that might be directly relevant to the user's question:
{context}

With this context, please answer the user's question. Be accurate, relevant and complete.
"""

In [78]:
# In the context, include the source of the chunk

def make_rag_messages(question, history, chunks):
    context = "\n\n".join(f"Extract from {chunk.metadata['source']}:\n{chunk.page_content}" for chunk in chunks)
    system_prompt = SYSTEM_PROMPT.format(context=context)
    return [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": question}]

In [79]:
# def rewrite_query(question, history=[]):
#     """Rewrite the user's question to be a more specific question that is more likely to surface relevant content in the Knowledge Base."""
#     message = f"""
# You are in a conversation with a user, answering questions about the company Insurellm.
# You are about to look up information in a Knowledge Base to answer the user's question.

# This is the history of your conversation so far with the user:
# {history}

# And this is the user's current question:
# {question}

# Respond only with a single, refined question that you will use to search the Knowledge Base.
# It should be a VERY short specific question most likely to surface content. Focus on the question details.
# Don't mention the company name unless it's a general question about the company.
# IMPORTANT: Respond ONLY with the knowledgebase query, nothing else.
# """
#     response = completion(model=MODEL, messages=[{"role": "system", "content": message}])
#     return response.choices[0].message.content
import warnings
# from pydantic import PydanticWarning

# 1. Silences the messy serialization logs from crowding your output
warnings.filterwarnings("ignore", category=UserWarning, module="pydantic")

def rewrite_query(question, history=[]):
    """
    Rewrites the question to maximize semantic search accuracy with built-in rate limit handling.
    """
    message = f"""
You are an expert search-query optimizer for a RAG knowledge base.
Your job is to rewrite the user's latest question into a short, dense search query.

CRITICAL INSTRUCTIONS:
1. Strip out conversational filler words ("who won", "can you tell me about").
2. Focus ONLY on core keyword subjects and identifiers (e.g., "IIOTY award winner").
3. Do NOT add extra context like "awarded by Insurellm" unless explicitly mentioned.
4. Output ONLY the raw search query string. No markdown quotes, no preamble.

CONVERSATION HISTORY:
{history}

USER QUESTION:
{question}
"""
    while True:
        try:
            response = completion(model=MODEL, messages=[{"role": "system", "content": message}])
            return response.choices[0].message.content.strip()
            
        except RateLimitError as e:
            print("\n⚠️ Groq TPM Rate Limit reached during Query Rewriting. Pausing 50s...")
            time.sleep(50)
            continue

In [80]:
rewrite_query("Who won the IIOTY award?", [])

'IIOTY award winner'

In [81]:
# def answer_question(question: str, history: list[dict] = []) -> tuple[str, list]:
#     """
#     Answer a question using RAG and return the answer and the retrieved context
#     """
#     query = rewrite_query(question, history)
#     print(query)
#     chunks = fetch_context(query)
#     messages = make_rag_messages(question, history, chunks)
#     response = completion(model=MODEL, messages=messages)
#     return response.choices[0].message.content, chunks
def answer_question(question: str, history: list[dict] = []) -> tuple[str, list]:
    """
    Answer a question using an advanced RAG pipeline with dense context filtering.
    """
    # 1. Optimize the incoming user query
    query = rewrite_query(question, history)
    print(f"🎯 Optimized Query: {query}")
    
    # 2. Retrieve all 20 unranked candidates and run them through your resilient re-ranker
    all_chunks = fetch_context(query)
    
    # 3. 🌟 CRITICAL FIX: Only pass the top 3 best re-ranked chunks to the generation engine!
    # This instantly drops your prompt payload from 5,433 tokens down to ~1,200 tokens.
    generation_chunks = all_chunks[:5] 
    
    # 4. Build the final prompt payload with the filtered chunks
    messages = make_rag_messages(question, history, generation_chunks)
    
    # 5. Generate the crisp, accurate response
    print("🤖 Insurellm is compiling the final answer...")
    response = completion(model=MODEL, messages=messages)
    
    # Return the clean text response alongside the full chunk list for debugging/sources
    return response.choices[0].message.content, all_chunks

In [82]:
answer_question("Who won the IIOTY award?", [])

🎯 Optimized Query: IIOTY award winner


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1157.93it/s]


🧩 Splitting 20 chunks into safe structural batches...
🏆 Running final validation sort on top contenders...

Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.


Provider List: https://docs.litellm.ai/docs/providers


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.


Provider List: https://docs.litellm.ai/docs/providers


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.


Provider List: https://docs.litellm.ai/docs/providers

🚨 [Fallback] Preserving default sequence on attempt 3.
🤖 Insurellm is compiling the final answer...

Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.


Provider Lis

RateLimitError: litellm.RateLimitError: RateLimitError: GroqException - {"error":{"message":"Rate limit reached for model `llama-3.1-8b-instant` in organization `org_01kkrkbwkxfmnb24xc48an8tmy` service tier `on_demand` on tokens per minute (TPM): Limit 6000, Used 5117, Requested 1056. Please try again in 1.73s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing","type":"tokens","code":"rate_limit_exceeded"}}


In [72]:
answer_question("Who went to Manchester University?", [])

🎯 Optimized Query: manchester university alumni


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3816.91it/s]


🧩 Splitting 20 chunks into smaller pools to protect Groq TPM ceilings...
   Sorting Batch 1/2...
   Sorting Batch 2/2...

Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.


Provider List: https://docs.litellm.ai/docs/providers


⚠️ Groq TPM Rate Limit reached during re-ranking. Pausing 65s for full reset...
🏆 Running final validation sort on top contenders...

Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.


Provider List: https://docs.litellm.ai/docs/providers


⚠️ Groq TPM Rate Limit reached during re-ranking. Pausing 65s for full reset...
🤖 Insurellm is compiling the final answer...

Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.


Provider List: https://docs.litellm.ai/docs/pro

RateLimitError: litellm.RateLimitError: RateLimitError: GroqException - {"error":{"message":"Rate limit reached for model `llama-3.1-8b-instant` in organization `org_01kkrkbwkxfmnb24xc48an8tmy` service tier `on_demand` on tokens per day (TPD): Limit 500000, Used 499147, Requested 1138. Please try again in 49.248s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing","type":"tokens","code":"rate_limit_exceeded"}}
